In [2]:
import csv
from pathlib import Path
import os
import numpy as np
import pandas as pd
import scanpy as sc
import scanpy.external as sce
import anndata as ad
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.lines import Line2D
from scipy import sparse
import harmonypy as hm


In [3]:
output_path = Path("/p2/zulab/jtian/data/SA/06_calculateConcentration/output_CAST_Kmeans_seperate/")
output_path.mkdir(parents=True, exist_ok=True)

In [6]:
base_dir = Path('/p2/zulab/jtian/data/SA/06_calculateConcentration/output_CAST_Leiden_seperate/CAST_MARK_by_sample')
sample_names = ['ctrl1', 'ctrl2', 'ctrl3', 'dn1', 'dn2', 'dn3']
sample_data = {}
for sample in sample_names:
    npz_path = base_dir / sample / 'demo1_kmeans_labels_k20.npz'
    loaded = np.load(npz_path, allow_pickle=True)
    sample_data[sample] = {key: loaded[key] for key in loaded.files}
    

In [17]:
timepoints = [0, 15, 30, 45, 60]
sample_clusters = {}

for sample_name in sample_names:
    arrays_in_order = [sample_data[sample_name][f'{sample_name}_{tp}'] for tp in timepoints]
    sample_clusters[sample_name] = np.concatenate(arrays_in_order)
    globals()[f'{sample_name}_cluster'] = sample_clusters[sample_name]

ctrl_cluster = np.concatenate([
    sample_clusters['ctrl1'],
    sample_clusters['ctrl2'],
    sample_clusters['ctrl3'],
])

dn_cluster = np.concatenate([
    sample_clusters['dn1'],
    sample_clusters['dn2'],
    sample_clusters['dn3'],
])





In [25]:
ctrl_merged=pd.read_csv("/p2/zulab/jtian/data/SA/06_calculateConcentration/output_PCA_Harmony_Leiden/ctrlIntensity_merged.csv",sep=';',index_col=0)
dn_merged=pd.read_csv("/p2/zulab/jtian/data/SA/06_calculateConcentration/output_PCA_Harmony_Leiden/dnIntensity_merged.csv",sep=';',index_col=0)



In [14]:
ctrl_merged

,Unnamed: 0,57.0346,58.0302,59.0139,67.0191,68.0143,68.9977,69.0346,70.0061,70.0299,...,858.5247,859.534,860.536,861.5477,862.552,863.5618,864.5684,882.5853,884.5423,888.5584
0,ctrl1_0_pixel1,0.000000,0.000000,0.261729,0.000000,0.234178,0.551008,1.418844,0.000000,0.000000,...,0.000000,0.103314,0.000000,0.192853,0.420143,0.172190,0.061988,0.110202,0.123977,0.082651
1,ctrl1_0_pixel2,0.000000,0.000000,0.140146,0.414068,0.203849,0.000000,0.847247,0.000000,0.872728,...,0.197479,0.299403,0.000000,0.121035,0.000000,0.070073,0.089184,0.000000,0.165627,0.471400
2,ctrl1_0_pixel3,0.000000,0.258983,0.000000,0.000000,0.267077,0.000000,0.000000,0.000000,0.922628,...,0.000000,0.000000,0.453221,0.000000,0.485594,0.000000,0.000000,0.000000,0.000000,0.226610
3,ctrl1_0_pixel4,0.201059,0.000000,0.000000,0.495467,0.359034,0.258505,0.517010,0.000000,0.883225,...,0.157975,0.229782,0.093349,0.409299,0.258505,0.215421,0.330312,0.452383,0.215421,0.000000
4,ctrl1_0_pixel5,0.577206,0.000000,0.849775,0.000000,0.416871,0.000000,0.665390,0.000000,0.665390,...,0.240502,0.000000,0.176368,0.168352,0.000000,0.000000,0.176368,0.144301,0.128268,0.368770
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
497831,ctrl3_60_pixel50660,2.105920,0.399399,10.329899,2.178538,2.341928,3.521969,2.596091,2.995489,4.393384,...,3.612742,5.627889,0.000000,5.754970,2.251156,3.558278,1.670212,2.723172,2.033302,1.343432
497832,ctrl3_60_pixel50661,2.115093,0.459090,16.691198,1.278893,2.033112,2.541391,6.427259,3.033273,5.246742,...,3.115253,4.640087,0.000000,6.033753,2.902104,3.672719,2.623371,1.869152,1.737983,2.180677
497833,ctrl3_60_pixel50662,1.906487,2.017329,14.121301,2.793224,4.633206,2.815393,6.894387,1.529623,4.965732,...,4.588869,5.918976,5.054406,8.623526,7.470767,4.655374,2.083834,1.950823,2.194676,1.950823
497834,ctrl3_60_pixel50663,1.522979,3.635498,7.148176,0.000000,2.947701,4.323295,11.397779,0.000000,8.941361,...,5.133913,11.299522,7.393818,9.604594,13.166400,7.614895,4.102218,6.091916,3.488113,2.751188


In [20]:
dn_merged

,Unnamed: 0,57.0346,58.0302,59.0139,67.0191,68.0143,68.9977,69.0346,70.0061,70.0299,...,862.552,863.5618,864.5684,884.5423,888.5673,918.5629,919.5555,921.5991,923.615,946.5988
0,dn1_0_pixel1,0.000000,0.712215,0.000000,0.000000,1.035949,0.684466,0.434729,0.000000,0.786211,...,0.000000,0.101745,0.203490,0.000000,0.000000,0.083246,0.000000,0.000000,0.000000,0.314484
1,dn1_0_pixel2,0.000000,1.836354,3.292773,1.308666,2.617332,2.385150,2.596225,0.000000,9.160663,...,0.970946,0.000000,0.000000,0.253290,0.337720,0.000000,0.675441,0.274398,0.295505,0.844301
2,dn1_0_pixel3,0.000000,1.547361,0.000000,1.246902,0.480733,0.961467,0.000000,0.435665,3.365134,...,0.000000,0.000000,0.540825,0.525802,0.240367,0.165252,0.000000,0.000000,0.420642,0.781192
3,dn1_0_pixel4,0.830629,0.540409,0.000000,0.000000,1.250948,0.000000,0.490371,0.230174,1.491129,...,0.350265,0.000000,0.180136,0.470356,0.370280,0.000000,0.570432,0.320243,0.000000,1.401061
4,dn1_0_pixel5,0.000000,0.000000,0.000000,0.202554,0.920700,1.031184,0.000000,0.000000,1.086427,...,0.000000,0.119691,0.082863,0.174933,0.239382,0.220968,0.000000,0.000000,0.211761,1.629640
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
605568,dn3_60_pixel58565,0.000000,1.639123,10.191071,1.971699,1.330303,6.200162,9.454654,1.995455,9.787229,...,5.178680,5.630032,2.399297,2.755628,0.000000,6.247673,2.470563,1.591613,2.826894,1.971699
605569,dn3_60_pixel58566,0.000000,2.189291,9.793167,0.801437,1.446496,5.101829,5.121377,0.723248,5.101829,...,3.362125,2.892991,2.267480,1.446496,1.896082,1.974271,2.091555,2.384763,3.850806,3.323031
605570,dn3_60_pixel58567,0.796473,1.655414,7.917875,1.811585,2.186395,2.326949,3.263976,2.311332,3.982363,...,2.326949,4.482111,1.546094,2.186395,0.000000,1.358689,1.155666,2.077076,2.217630,1.499243
605571,dn3_60_pixel58568,0.455358,0.683037,6.342484,1.317285,2.016585,4.325900,3.886804,3.642863,4.651155,...,5.025199,3.398921,1.967796,1.512439,1.479913,1.723855,2.715884,2.439417,1.707592,1.252234


In [23]:
ctrl_merged.insert(0, "cluster", ctrl_cluster)
dn_merged.insert(0, "cluster", dn_cluster)

In [ ]:
ctrl_merged

,cluster,57.0346,58.0302,59.0139,67.0191,68.0143,68.9977,69.0346,70.0061,70.0299,...,858.5247,859.534,860.536,861.5477,862.552,863.5618,864.5684,882.5853,884.5423,888.5584
ctrl1_0_pixel1,0,0.000000,0.000000,0.261729,0.000000,0.234178,0.551008,1.418844,0.000000,0.000000,...,0.000000,0.103314,0.000000,0.192853,0.420143,0.172190,0.061988,0.110202,0.123977,0.082651
ctrl1_0_pixel2,0,0.000000,0.000000,0.140146,0.414068,0.203849,0.000000,0.847247,0.000000,0.872728,...,0.197479,0.299403,0.000000,0.121035,0.000000,0.070073,0.089184,0.000000,0.165627,0.471400
ctrl1_0_pixel3,0,0.000000,0.258983,0.000000,0.000000,0.267077,0.000000,0.000000,0.000000,0.922628,...,0.000000,0.000000,0.453221,0.000000,0.485594,0.000000,0.000000,0.000000,0.000000,0.226610
ctrl1_0_pixel4,0,0.201059,0.000000,0.000000,0.495467,0.359034,0.258505,0.517010,0.000000,0.883225,...,0.157975,0.229782,0.093349,0.409299,0.258505,0.215421,0.330312,0.452383,0.215421,0.000000
ctrl1_0_pixel5,0,0.577206,0.000000,0.849775,0.000000,0.416871,0.000000,0.665390,0.000000,0.665390,...,0.240502,0.000000,0.176368,0.168352,0.000000,0.000000,0.176368,0.144301,0.128268,0.368770
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ctrl3_60_pixel50660,17,2.105920,0.399399,10.329899,2.178538,2.341928,3.521969,2.596091,2.995489,4.393384,...,3.612742,5.627889,0.000000,5.754970,2.251156,3.558278,1.670212,2.723172,2.033302,1.343432
ctrl3_60_pixel50661,17,2.115093,0.459090,16.691198,1.278893,2.033112,2.541391,6.427259,3.033273,5.246742,...,3.115253,4.640087,0.000000,6.033753,2.902104,3.672719,2.623371,1.869152,1.737983,2.180677
ctrl3_60_pixel50662,17,1.906487,2.017329,14.121301,2.793224,4.633206,2.815393,6.894387,1.529623,4.965732,...,4.588869,5.918976,5.054406,8.623526,7.470767,4.655374,2.083834,1.950823,2.194676,1.950823
ctrl3_60_pixel50663,17,1.522979,3.635498,7.148176,0.000000,2.947701,4.323295,11.397779,0.000000,8.941361,...,5.133913,11.299522,7.393818,9.604594,13.166400,7.614895,4.102218,6.091916,3.488113,2.751188


In [25]:
dn_merged

,cluster,57.0346,58.0302,59.0139,67.0191,68.0143,68.9977,69.0346,70.0061,70.0299,...,862.552,863.5618,864.5684,884.5423,888.5673,918.5629,919.5555,921.5991,923.615,946.5988
dn1_0_pixel1,12,0.000000,0.712215,0.000000,0.000000,1.035949,0.684466,0.434729,0.000000,0.786211,...,0.000000,0.101745,0.203490,0.000000,0.000000,0.083246,0.000000,0.000000,0.000000,0.314484
dn1_0_pixel2,3,0.000000,1.836354,3.292773,1.308666,2.617332,2.385150,2.596225,0.000000,9.160663,...,0.970946,0.000000,0.000000,0.253290,0.337720,0.000000,0.675441,0.274398,0.295505,0.844301
dn1_0_pixel3,3,0.000000,1.547361,0.000000,1.246902,0.480733,0.961467,0.000000,0.435665,3.365134,...,0.000000,0.000000,0.540825,0.525802,0.240367,0.165252,0.000000,0.000000,0.420642,0.781192
dn1_0_pixel4,3,0.830629,0.540409,0.000000,0.000000,1.250948,0.000000,0.490371,0.230174,1.491129,...,0.350265,0.000000,0.180136,0.470356,0.370280,0.000000,0.570432,0.320243,0.000000,1.401061
dn1_0_pixel5,12,0.000000,0.000000,0.000000,0.202554,0.920700,1.031184,0.000000,0.000000,1.086427,...,0.000000,0.119691,0.082863,0.174933,0.239382,0.220968,0.000000,0.000000,0.211761,1.629640
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
dn3_60_pixel58565,5,0.000000,1.639123,10.191071,1.971699,1.330303,6.200162,9.454654,1.995455,9.787229,...,5.178680,5.630032,2.399297,2.755628,0.000000,6.247673,2.470563,1.591613,2.826894,1.971699
dn3_60_pixel58566,5,0.000000,2.189291,9.793167,0.801437,1.446496,5.101829,5.121377,0.723248,5.101829,...,3.362125,2.892991,2.267480,1.446496,1.896082,1.974271,2.091555,2.384763,3.850806,3.323031
dn3_60_pixel58567,5,0.796473,1.655414,7.917875,1.811585,2.186395,2.326949,3.263976,2.311332,3.982363,...,2.326949,4.482111,1.546094,2.186395,0.000000,1.358689,1.155666,2.077076,2.217630,1.499243
dn3_60_pixel58568,5,0.455358,0.683037,6.342484,1.317285,2.016585,4.325900,3.886804,3.642863,4.651155,...,5.025199,3.398921,1.967796,1.512439,1.479913,1.723855,2.715884,2.439417,1.707592,1.252234


In [28]:
ctrl_merged.to_csv(output_path/'ctrlIntensityCluster.csv')
dn_merged.to_csv(output_path/'dnIntensityCluster.csv')

In [29]:
ctrl_mz_cols = [c for c in ctrl_merged.columns if c != "cluster"]
ctrl_mz_cols = sorted(ctrl_mz_cols, key=lambda x: float(x))
ctrl_merged = ctrl_merged[["cluster"] + ctrl_mz_cols]

ctrl_tmp = ctrl_merged.copy()
ctrl_tmp["group"] = ctrl_tmp.index.to_series().str.replace(r"_pixel\d+$", "", regex=True)
ctrl_cluster_mean = ctrl_tmp.groupby(
    ["group", "cluster"],
    sort=False
)[ctrl_mz_cols].mean()
ctrl_cluster_mean.index = [
    f"{group}_cluster{cluster}"
    for group, cluster in ctrl_cluster_mean.index
]
ctrl_sort_df = ctrl_cluster_mean.copy()
ctrl_parts = ctrl_sort_df.index.to_series().str.extract(
    r"^ctrl(\d+)_(\d+)_cluster(\d+)$"
)
ctrl_sort_df["_sample_num"] = ctrl_parts[0].astype(int)
ctrl_sort_df["_time"] = ctrl_parts[1].astype(int)
ctrl_sort_df["_cluster"] = ctrl_parts[2].astype(int)
ctrl_sort_df = ctrl_sort_df.sort_values(
    by=["_sample_num", "_cluster", "_time"],
    kind="stable"
)
ctrl_cluster_mean = ctrl_sort_df.drop(
    columns=["_sample_num", "_time", "_cluster"]
)
print(ctrl_cluster_mean.iloc[:10,:5])

                    57.0346   58.0302   59.0139   67.0191   68.0143
ctrl1_0_cluster0   0.599185  1.007506  3.835550  1.029358  2.092574
ctrl1_15_cluster0  0.693865  1.100687  4.574478  1.159652  1.901231
ctrl1_30_cluster0  0.718842  1.038590  4.873938  1.177825  1.619052
ctrl1_45_cluster0  0.751595  1.078817  5.326885  1.282269  1.639043
ctrl1_60_cluster0  0.787080  1.072928  5.523687  1.372648  1.591864
ctrl1_0_cluster1   0.629570  0.949219  3.596401  1.057952  2.023340
ctrl1_15_cluster1  0.677066  1.068015  3.974989  1.095449  1.626973
ctrl1_30_cluster1  0.676748  1.029907  4.298627  1.243694  1.330542
ctrl1_45_cluster1  0.697910  1.019193  4.597609  1.299724  1.415132
ctrl1_60_cluster1  0.714647  1.039778  4.780010  1.543425  1.319907


In [30]:
dn_mz_cols = [c for c in dn_merged.columns if c != "cluster"]
dn_mz_cols = sorted(dn_mz_cols, key=lambda x: float(x))
dn_merged = dn_merged[["cluster"] + dn_mz_cols]

dn_tmp = dn_merged.copy()
dn_tmp["group"] = dn_tmp.index.to_series().str.replace(r"_pixel\d+$", "", regex=True)
dn_cluster_mean = dn_tmp.groupby(
    ["group", "cluster"],
    sort=False
)[dn_mz_cols].mean()
dn_cluster_mean.index = [
    f"{group}_cluster{cluster}"
    for group, cluster in dn_cluster_mean.index
]

dn_sort_df = dn_cluster_mean.copy()

dn_parts = dn_sort_df.index.to_series().str.extract(
    r"^dn(\d+)_(\d+)_cluster(\d+)$"
)

dn_sort_df["_sample_num"] = dn_parts[0].astype(int)
dn_sort_df["_time"] = dn_parts[1].astype(int)
dn_sort_df["_cluster"] = dn_parts[2].astype(int)

dn_sort_df = dn_sort_df.sort_values(
    by=["_sample_num", "_cluster", "_time"],
    kind="stable"
)

dn_cluster_mean = dn_sort_df.drop(
    columns=["_sample_num", "_time", "_cluster"]
)
print(dn_cluster_mean.iloc[:10,:5])

                  57.0346   58.0302   59.0139   67.0191   68.0143
dn1_0_cluster0   0.696484  0.940211  5.217174  1.130012  2.250896
dn1_15_cluster0  0.762053  1.035588  4.798021  1.288879  1.773062
dn1_30_cluster0  0.805856  1.138010  5.313955  1.437479  1.855939
dn1_45_cluster0  0.838536  1.036011  5.430212  1.468211  1.725029
dn1_60_cluster0  0.762440  0.991175  5.376657  1.476172  1.475774
dn1_0_cluster1   0.680682  1.100953  4.685559  1.117559  2.267021
dn1_15_cluster1  0.732460  1.190690  4.784731  1.259025  1.811230
dn1_30_cluster1  0.811983  1.309803  5.365309  1.358640  1.890336
dn1_45_cluster1  0.830216  1.294059  5.609156  1.462812  1.795439
dn1_60_cluster1  0.804169  1.155345  5.483847  1.475314  1.571948


In [31]:
ctrl_cluster_mean.to_csv(output_path/'ctrlIntensityMeanByCluster.csv')
dn_cluster_mean.to_csv(output_path/'dnIntensityMeanByCluster.csv')

In [40]:
def linreg_stats(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    ok = np.isfinite(x) & np.isfinite(y)
    x = x[ok]; y = y[ok]
    n = len(x)
    if n < 2:
        return np.nan, np.nan, np.nan, np.nan

    xm = x.mean()
    ym = y.mean()
    dx = x - xm
    dy = y - ym

    varx = np.sum(dx * dx)
    if varx == 0:
        return np.nan, np.nan, np.nan, np.nan

    slope = np.sum(dx * dy) / varx
    intercept = ym - slope * xm

    yhat = slope * x + intercept
    ss_res = np.sum((y - yhat) ** 2)
    ss_tot = np.sum((y - ym) ** 2)
    r2 = 1 - ss_res / ss_tot if ss_tot != 0 else np.nan

    # y=0 时 x 截距：x0 = -b/a；取绝对值
    x0_abs = np.abs(-intercept / slope) if slope != 0 else np.nan
    return slope, intercept, r2, x0_abs

In [41]:
def build_linreg_result(df):
    """
    df 的行名格式：
    ctrl1_0_cluster0 / dn1_15_cluster2 这类
    列名为 m/z
    """
    tmp = df.copy()

    # 解析行名：样本号、时间、cluster
    parts = tmp.index.to_series().str.extract(
        r"^([A-Za-z]+\d+)_(\d+)_cluster(\d+)$"
    )

    tmp["_sample"] = parts[0]
    tmp["_time"] = parts[1].astype(int)
    tmp["_cluster"] = parts[2].astype(int)

    # m/z 列
    mz_cols = [c for c in tmp.columns if c not in ["_sample", "_time", "_cluster"]]
    mz_cols = sorted(mz_cols, key=lambda x: float(x))

    result_rows = []

    # 按 sample + cluster 分组
    for (sample, cluster), sub in tmp.groupby(["_sample", "_cluster"], sort=False):
        sub = sub.sort_values("_time")

        # 这里 x 实际上就是 [0,15,30,45,60]
        x = sub["_time"].to_numpy(dtype=float)

        for mz in mz_cols:
            y = sub[mz].to_numpy(dtype=float)
            slope, intercept, r2, x0_abs = linreg_stats(x, y)

            result_rows.append({
                "sample": sample,
                "mz": float(mz),
                "cluster": int(cluster),
                "slope": slope,
                "intercept": intercept,
                "r2": r2,
                "x0_abs": x0_abs
            })

    result_df = pd.DataFrame(result_rows)

    # 排序：先 sample，再 cluster，再 mz
    parts2 = result_df["sample"].str.extract(r"^([A-Za-z]+)(\d+)$")
    result_df["_prefix"] = parts2[0]
    result_df["_sample_num"] = parts2[1].astype(int)

    result_df = result_df.sort_values(
        by=["_prefix", "_sample_num", "cluster", "mz"],
        kind="stable"
    ).drop(columns=["_prefix", "_sample_num"])

    result_df = result_df.reset_index(drop=True)
    return result_df

In [34]:
ctrl_linreg_result = build_linreg_result(ctrl_cluster_mean)
dn_linreg_result = build_linreg_result(dn_cluster_mean)

print(ctrl_linreg_result.head())
print(dn_linreg_result.head())

  sample       mz  cluster     slope  intercept        r2       x0_abs
0  ctrl1  57.0346        0  0.002890   0.623410  0.926272   215.702921
1  ctrl1  58.0302        0  0.000726   1.037911  0.220312  1428.660577
2  ctrl1  59.0139        0  0.027525   4.001171  0.955386   145.367426
3  ctrl1  67.0191        0  0.005395   1.042511  0.966932   193.249259
4  ctrl1  68.0143        0 -0.008424   2.021474  0.827585   239.964532
  sample       mz  cluster     slope  intercept        r2       x0_abs
0    dn1  57.0346        0  0.001389   0.731395  0.378941   526.449848
1    dn1  58.0302        0  0.000682   1.007729  0.049213  1476.873135
2    dn1  59.0139        0  0.006341   5.036972  0.354266   794.344396
3    dn1  67.0191        0  0.005811   1.185820  0.852141   204.064365
4    dn1  68.0143        0 -0.010655   2.135796  0.806844   200.446523


In [35]:
ctrl_r2_mean = (
    ctrl_linreg_result
    .groupby(["sample", "mz"], as_index=False)["r2"]
    .mean()
    .rename(columns={"r2": "mean_r2"})
)

ctrl_good = ctrl_r2_mean[ctrl_r2_mean["mean_r2"] >= 0.9].copy()

# 每个样本筛出来多少个代谢物
ctrl_count = (
    ctrl_good
    .groupby("sample", as_index=False)
    .size()
    .rename(columns={"size": "n_metabolites_r2_ge_0.9"})
)

print("ctrl_good:")
print(ctrl_good.head())

print("ctrl_count:")
print(ctrl_count)


dn_r2_mean = (
    dn_linreg_result
    .groupby(["sample", "mz"], as_index=False)["r2"]
    .mean()
    .rename(columns={"r2": "mean_r2"})
)

dn_good = dn_r2_mean[dn_r2_mean["mean_r2"] >= 0.9].copy()

dn_count = (
    dn_good
    .groupby("sample", as_index=False)
    .size()
    .rename(columns={"size": "n_metabolites_r2_ge_0.9"})
)

print("dn_good:")
print(dn_good.head())

print("dn_count:")
print(dn_count)

ctrl_good:
   sample       mz   mean_r2
2   ctrl1  59.0139  0.954229
10  ctrl1  71.0503  0.900884
14  ctrl1  73.0294  0.964942
15  ctrl1  74.0246  0.965059
17  ctrl1  77.0394  0.932080
ctrl_count:
  sample  n_metabolites_r2_ge_0.9
0  ctrl1                      242
1  ctrl2                       94
2  ctrl3                      359
dn_good:
    sample        mz   mean_r2
35     dn1   89.0246  0.913581
76     dn1  110.9758  0.921878
117    dn1  126.9048  0.961321
152    dn1  136.8625  0.922013
174    dn1  145.0614  0.900829
dn_count:
  sample  n_metabolites_r2_ge_0.9
0    dn1                       82
1    dn2                      133
2    dn3                      101


In [32]:
coordsLabelIntensity=pd.read_csv("/p2/zulab/jtian/data/SA/06_calculateConcentration/output_calculateByClusterLR/coordsLabelIntensity.csv")


In [37]:
coordsLabelIntensity

,Unnamed: 0,sample,cell_label,57.0346,58.0296,59.0139,67.0191,71.0137,71.0503,72.0091,...,856.5052,859.5297,860.536,863.5618,864.5684,882.5853,921.5991,923.615,985.6046,986.6105
0,1,0,5,0.208880,0.254289,0.499496,0.399597,1.607469,0.127144,0.962665,...,0.363270,0.481332,0.163471,0.000000,0.435924,0.308779,0.000000,0.000000,0.163471,0.000000
1,2,0,5,0.000000,0.223221,0.480784,0.257563,2.481191,0.000000,1.210547,...,0.712591,0.154538,0.283319,0.394930,0.463614,0.257563,0.154538,0.000000,0.120196,0.412101
2,3,0,5,0.000000,0.000000,1.175326,0.720719,0.698543,0.000000,0.155232,...,0.177408,0.155232,0.221760,0.410255,0.000000,0.000000,0.000000,0.232848,0.654191,0.343727
3,4,0,5,0.152048,0.000000,0.557510,0.000000,0.726453,0.000000,0.278755,...,0.084471,0.000000,0.194284,0.304097,0.464592,0.312544,0.278755,0.000000,0.000000,0.000000
4,5,0,5,0.000000,0.000000,0.725190,0.382195,2.587164,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.088199,0.244997,0.244997,0.195997,0.000000,0.391995,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
247356,247357,60,5,2.105920,0.399399,10.329899,2.178538,39.013979,1.216350,4.302612,...,4.393384,5.627889,0.000000,3.558278,1.670212,2.723172,1.379741,1.760985,3.739823,2.360082
247357,247358,60,5,2.115093,0.459090,16.691198,1.278893,49.745672,2.672559,4.115413,...,3.148045,4.640087,0.000000,3.672719,2.623371,1.869152,1.721587,0.950972,4.230186,2.229865
247358,247359,60,5,1.906487,2.017329,14.121301,2.793224,41.831861,0.532043,6.916556,...,3.591289,5.918976,5.054406,4.655374,2.083834,1.950823,2.394192,1.773476,5.409101,2.726719
247359,247360,60,5,1.522979,3.635498,7.148176,0.000000,30.975429,1.007131,2.554675,...,8.622027,11.299522,7.393818,7.614895,4.102218,6.091916,2.309033,1.940570,5.035657,2.972266


In [33]:
coordsLabelIntensity = coordsLabelIntensity.drop(columns=coordsLabelIntensity.columns[0])

In [39]:
coordsLabelIntensity

,sample,cell_label,57.0346,58.0296,59.0139,67.0191,71.0137,71.0503,72.0091,72.017,...,856.5052,859.5297,860.536,863.5618,864.5684,882.5853,921.5991,923.615,985.6046,986.6105
0,0,5,0.208880,0.254289,0.499496,0.399597,1.607469,0.127144,0.962665,0.354188,...,0.363270,0.481332,0.163471,0.000000,0.435924,0.308779,0.000000,0.000000,0.163471,0.000000
1,0,5,0.000000,0.223221,0.480784,0.257563,2.481191,0.000000,1.210547,0.000000,...,0.712591,0.154538,0.283319,0.394930,0.463614,0.257563,0.154538,0.000000,0.120196,0.412101
2,0,5,0.000000,0.000000,1.175326,0.720719,0.698543,0.000000,0.155232,0.000000,...,0.177408,0.155232,0.221760,0.410255,0.000000,0.000000,0.000000,0.232848,0.654191,0.343727
3,0,5,0.152048,0.000000,0.557510,0.000000,0.726453,0.000000,0.278755,0.143601,...,0.084471,0.000000,0.194284,0.304097,0.464592,0.312544,0.278755,0.000000,0.000000,0.000000
4,0,5,0.000000,0.000000,0.725190,0.382195,2.587164,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.088199,0.244997,0.244997,0.195997,0.000000,0.391995,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
247356,60,5,2.105920,0.399399,10.329899,2.178538,39.013979,1.216350,4.302612,0.000000,...,4.393384,5.627889,0.000000,3.558278,1.670212,2.723172,1.379741,1.760985,3.739823,2.360082
247357,60,5,2.115093,0.459090,16.691198,1.278893,49.745672,2.672559,4.115413,0.000000,...,3.148045,4.640087,0.000000,3.672719,2.623371,1.869152,1.721587,0.950972,4.230186,2.229865
247358,60,5,1.906487,2.017329,14.121301,2.793224,41.831861,0.532043,6.916556,0.000000,...,3.591289,5.918976,5.054406,4.655374,2.083834,1.950823,2.394192,1.773476,5.409101,2.726719
247359,60,5,1.522979,3.635498,7.148176,0.000000,30.975429,1.007131,2.554675,0.000000,...,8.622027,11.299522,7.393818,7.614895,4.102218,6.091916,2.309033,1.940570,5.035657,2.972266


In [34]:
coordsLabelIntensity=coordsLabelIntensity.rename(columns={'cell_label':'cluster'})

In [22]:
coordsLabelIntensity['cluster']

0         5
1         5
2         5
3         5
4         5
         ..
247356    5
247357    5
247358    5
247359    5
247360    5
Name: cluster, Length: 247361, dtype: int64

In [35]:
coordsLabelIntensity.index=('ctrl3_'+coordsLabelIntensity['sample'].astype(str)+'_pixel'+coordsLabelIntensity.index.to_series().astype(str))

In [24]:
coordsLabelIntensity

,sample,cluster,57.0346,58.0296,59.0139,67.0191,71.0137,71.0503,72.0091,72.017,...,856.5052,859.5297,860.536,863.5618,864.5684,882.5853,921.5991,923.615,985.6046,986.6105
ctrl3_0_pixel0,0,5,0.208880,0.254289,0.499496,0.399597,1.607469,0.127144,0.962665,0.354188,...,0.363270,0.481332,0.163471,0.000000,0.435924,0.308779,0.000000,0.000000,0.163471,0.000000
ctrl3_0_pixel1,0,5,0.000000,0.223221,0.480784,0.257563,2.481191,0.000000,1.210547,0.000000,...,0.712591,0.154538,0.283319,0.394930,0.463614,0.257563,0.154538,0.000000,0.120196,0.412101
ctrl3_0_pixel2,0,5,0.000000,0.000000,1.175326,0.720719,0.698543,0.000000,0.155232,0.000000,...,0.177408,0.155232,0.221760,0.410255,0.000000,0.000000,0.000000,0.232848,0.654191,0.343727
ctrl3_0_pixel3,0,5,0.152048,0.000000,0.557510,0.000000,0.726453,0.000000,0.278755,0.143601,...,0.084471,0.000000,0.194284,0.304097,0.464592,0.312544,0.278755,0.000000,0.000000,0.000000
ctrl3_0_pixel4,0,5,0.000000,0.000000,0.725190,0.382195,2.587164,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.088199,0.244997,0.244997,0.195997,0.000000,0.391995,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ctrl3_60_pixel247356,60,5,2.105920,0.399399,10.329899,2.178538,39.013979,1.216350,4.302612,0.000000,...,4.393384,5.627889,0.000000,3.558278,1.670212,2.723172,1.379741,1.760985,3.739823,2.360082
ctrl3_60_pixel247357,60,5,2.115093,0.459090,16.691198,1.278893,49.745672,2.672559,4.115413,0.000000,...,3.148045,4.640087,0.000000,3.672719,2.623371,1.869152,1.721587,0.950972,4.230186,2.229865
ctrl3_60_pixel247358,60,5,1.906487,2.017329,14.121301,2.793224,41.831861,0.532043,6.916556,0.000000,...,3.591289,5.918976,5.054406,4.655374,2.083834,1.950823,2.394192,1.773476,5.409101,2.726719
ctrl3_60_pixel247359,60,5,1.522979,3.635498,7.148176,0.000000,30.975429,1.007131,2.554675,0.000000,...,8.622027,11.299522,7.393818,7.614895,4.102218,6.091916,2.309033,1.940570,5.035657,2.972266


In [36]:
coordsLabelIntensity=coordsLabelIntensity.drop(columns=['sample'])

In [37]:
ctrl_merged=coordsLabelIntensity.copy()

In [38]:
ctrl_merged

,cluster,57.0346,58.0296,59.0139,67.0191,71.0137,71.0503,72.0091,72.017,73.0294,...,856.5052,859.5297,860.536,863.5618,864.5684,882.5853,921.5991,923.615,985.6046,986.6105
ctrl3_0_pixel0,5,0.208880,0.254289,0.499496,0.399597,1.607469,0.127144,0.962665,0.354188,0.563068,...,0.363270,0.481332,0.163471,0.000000,0.435924,0.308779,0.000000,0.000000,0.163471,0.000000
ctrl3_0_pixel1,5,0.000000,0.223221,0.480784,0.257563,2.481191,0.000000,1.210547,0.000000,0.875714,...,0.712591,0.154538,0.283319,0.394930,0.463614,0.257563,0.154538,0.000000,0.120196,0.412101
ctrl3_0_pixel2,5,0.000000,0.000000,1.175326,0.720719,0.698543,0.000000,0.155232,0.000000,0.000000,...,0.177408,0.155232,0.221760,0.410255,0.000000,0.000000,0.000000,0.232848,0.654191,0.343727
ctrl3_0_pixel3,5,0.152048,0.000000,0.557510,0.000000,0.726453,0.000000,0.278755,0.143601,0.574405,...,0.084471,0.000000,0.194284,0.304097,0.464592,0.312544,0.278755,0.000000,0.000000,0.000000
ctrl3_0_pixel4,5,0.000000,0.000000,0.725190,0.382195,2.587164,0.000000,0.000000,0.000000,1.460180,...,0.000000,0.000000,0.000000,0.088199,0.244997,0.244997,0.195997,0.000000,0.391995,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ctrl3_60_pixel247356,5,2.105920,0.399399,10.329899,2.178538,39.013979,1.216350,4.302612,0.000000,8.641533,...,4.393384,5.627889,0.000000,3.558278,1.670212,2.723172,1.379741,1.760985,3.739823,2.360082
ctrl3_60_pixel247357,5,2.115093,0.459090,16.691198,1.278893,49.745672,2.672559,4.115413,0.000000,20.085184,...,3.148045,4.640087,0.000000,3.672719,2.623371,1.869152,1.721587,0.950972,4.230186,2.229865
ctrl3_60_pixel247358,5,1.906487,2.017329,14.121301,2.793224,41.831861,0.532043,6.916556,0.000000,24.584808,...,3.591289,5.918976,5.054406,4.655374,2.083834,1.950823,2.394192,1.773476,5.409101,2.726719
ctrl3_60_pixel247359,5,1.522979,3.635498,7.148176,0.000000,30.975429,1.007131,2.554675,0.000000,14.713943,...,8.622027,11.299522,7.393818,7.614895,4.102218,6.091916,2.309033,1.940570,5.035657,2.972266


In [39]:
ctrl_mz_cols = [c for c in ctrl_merged.columns if c != "cluster"]
ctrl_mz_cols = sorted(ctrl_mz_cols, key=lambda x: float(x))
ctrl_merged = ctrl_merged[["cluster"] + ctrl_mz_cols]

ctrl_tmp = ctrl_merged.copy()
ctrl_tmp["group"] = ctrl_tmp.index.to_series().str.replace(r"_pixel\d+$", "", regex=True)
ctrl_cluster_mean = ctrl_tmp.groupby(
    ["group", "cluster"],
    sort=False
)[ctrl_mz_cols].mean()
ctrl_cluster_mean.index = [
    f"{group}_cluster{cluster}"
    for group, cluster in ctrl_cluster_mean.index
]
ctrl_sort_df = ctrl_cluster_mean.copy()
ctrl_parts = ctrl_sort_df.index.to_series().str.extract(
    r"^ctrl(\d+)_(\d+)_cluster(\d+)$"
)
ctrl_sort_df["_sample_num"] = ctrl_parts[0].astype(int)
ctrl_sort_df["_time"] = ctrl_parts[1].astype(int)
ctrl_sort_df["_cluster"] = ctrl_parts[2].astype(int)
ctrl_sort_df = ctrl_sort_df.sort_values(
    by=["_sample_num", "_cluster", "_time"],
    kind="stable"
)
ctrl_cluster_mean = ctrl_sort_df.drop(
    columns=["_sample_num", "_time", "_cluster"]
)
print(ctrl_cluster_mean.iloc[:10,:5])

                    57.0346   58.0296   59.0139   67.0191    71.0137
ctrl3_0_cluster0   0.672462  1.183939  4.577493  1.168264  20.045918
ctrl3_15_cluster0  0.788363  1.238375  5.199554  1.248263  21.114137
ctrl3_30_cluster0  0.834051  1.301432  5.674117  1.359843  23.042916
ctrl3_45_cluster0  0.864793  1.326989  6.019091  1.424356  24.481150
ctrl3_60_cluster0  0.898763  1.378686  6.369907  1.562330  26.040713
ctrl3_0_cluster1   0.752187  1.072616  4.595960  1.191684  19.786215
ctrl3_15_cluster1  0.807342  1.186582  5.284978  1.282130  21.371625
ctrl3_30_cluster1  0.837168  1.248545  5.631220  1.365701  22.506491
ctrl3_45_cluster1  0.836904  1.295224  5.870003  1.417393  23.995233
ctrl3_60_cluster1  0.870852  1.292029  6.313507  1.619172  25.315472


In [42]:
ctrl_linreg_result = build_linreg_result(ctrl_cluster_mean)

In [43]:
print(ctrl_linreg_result.head())

  sample       mz  cluster     slope  intercept        r2      x0_abs
0  ctrl3  57.0346        0  0.003527   0.705880  0.907794  200.142660
1  ctrl3  58.0296        0  0.003187   1.190263  0.985569  373.429464
2  ctrl3  59.0139        0  0.029362   4.687160  0.982371  159.631240
3  ctrl3  67.0191        0  0.006428   1.159766  0.988506  180.419492
4  ctrl3  71.0137        0  0.102377  19.873646  0.994644  194.121509


In [44]:
ctrl_r2_mean = (
    ctrl_linreg_result
    .groupby(["sample", "mz"], as_index=False)["r2"]
    .mean()
    .rename(columns={"r2": "mean_r2"})
)

ctrl_good = ctrl_r2_mean[ctrl_r2_mean["mean_r2"] >= 0.9].copy()

# 每个样本筛出来多少个代谢物
ctrl_count = (
    ctrl_good
    .groupby("sample", as_index=False)
    .size()
    .rename(columns={"size": "n_metabolites_r2_ge_0.9"})
)

print("ctrl_good:")
print(ctrl_good.head())

print("ctrl_count:")
print(ctrl_count)

ctrl_good:
  sample       mz   mean_r2
0  ctrl3  57.0346  0.916164
2  ctrl3  59.0139  0.952930
3  ctrl3  67.0191  0.965019
4  ctrl3  71.0137  0.944685
5  ctrl3  71.0503  0.947054
ctrl_count:
  sample  n_metabolites_r2_ge_0.9
0  ctrl3                      460
